In [2]:
import subprocess
import sys

packages = [
    "torch-geometric",
    "pandas",
    "numpy",
    "scikit-learn",
    "tqdm",
]

print("Installing packages...")
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import torch
import torch_geometric
print(f"PyTorch  : {torch.__version__}")
print(f"PyG      : {torch_geometric.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
print(f"pyg-lib  : {torch_geometric.typing.WITH_PYG_LIB}")
print(f"torch-sparse: {torch_geometric.typing.WITH_TORCH_SPARSE}")

Installing packages...
PyTorch  : 2.10.0+cu128
PyG      : 2.7.0
CUDA     : True
pyg-lib  : False
torch-sparse: False


In [3]:
import os
import json
import time
import warnings
from collections import defaultdict, deque
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder
from torch_geometric.data import Data
from torch_geometric.nn import GATConv
from tqdm.notebook import trange, tqdm

warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cuda


In [4]:
class Config:
    """
    Change DATASET_NAME to switch between datasets.
    Supported values (case-sensitive, must match filename prefix):
        LI-Small, HI-Small, LI-Medium, HI-Medium

    Colab free T4 memory guide:
        LI-Small  ~ 7M  transactions  -> fits comfortably
        HI-Small  ~ 5M  transactions  -> fits comfortably
        LI-Medium ~ 31M transactions  -> marginal, may OOM
        HI-Medium ~ 32M transactions  -> marginal, may OOM

    Recommended for free Colab: LI-Small or HI-Small
    """

    DATASET_NAME = "HI-Large"   # <-- change only this line to swap datasets

    # Colab path after mounting Kaggle or uploading manually
    DATA_PATH = "/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/"

    # Model
    HIDDEN_DIM    = 64
    NUM_LAYERS    = 2
    NUM_HEADS     = 4       # GAT attention heads
    DROPOUT       = 0.3

    # Training
    EPOCHS                   = 50
    LEARNING_RATE            = 0.001
    WEIGHT_DECAY             = 1e-5
    MINORITY_CLASS_WEIGHT    = 10.0   # upweight laundering class
    EARLY_STOPPING_PATIENCE  = 10

    # Inference
    SUSPICION_THRESHOLD = 0.5   # probability threshold for flagging

    DEVICE = DEVICE

cfg = Config()
print(f"Dataset : {cfg.DATASET_NAME}")
print(f"Path    : {cfg.DATA_PATH}")
print(f"Device  : {cfg.DEVICE}")

Dataset : HI-Large
Path    : /kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/
Device  : cuda


In [6]:
import pandas as pd
import numpy as np
import os

def load_balanced_dataset(cfg):
    # Construct path for HI-Large
    base = os.path.join(cfg.DATA_PATH, cfg.DATASET_NAME)
    t_path = f"{base}_Trans.csv"
    
    print(f"Pulling and Balancing {cfg.DATASET_NAME}...")

    # 1. Collect ALL laundering cases first (they are rare)
    laundering_rows = []
    normal_sample_pool = []
    
    # Process in chunks to keep RAM usage low
    for chunk in pd.read_csv(t_path, chunksize=1000000):
        illicit = chunk[chunk['Is Laundering'] == 1]
        laundering_rows.append(illicit)
        
        # Take a 1% random sample of normal rows to choose from later
        licit = chunk[chunk['Is Laundering'] == 0].sample(frac=0.01)
        normal_sample_pool.append(licit)
        
    df_illicit = pd.concat(laundering_rows)
    df_licit_potential = pd.concat(normal_sample_pool)
    
    # 2. Match 1:1 ratio
    # We take exactly as many normal rows as there are laundering rows
    df_licit = df_licit_potential.sample(n=len(df_illicit), random_state=42)
    
    # 3. Final Balanced Dataframe
    balanced_df = pd.concat([df_illicit, df_licit]).sample(frac=1).reset_index(drop=True)
    
    # 4. Load Accounts
    accounts_df = pd.read_csv(f"{base}_accounts.csv")
    
    print(f"Successfully balanced! Total rows: {len(balanced_df)}")
    return balanced_df, accounts_df

# Replace the variables used in the rest of your notebook
trans_df, accounts_df = load_balanced_dataset(cfg)    

Pulling and Balancing HI-Large...
Successfully balanced! Total rows: 451092


In [ ]:
# # Run this cell only if you are uploading files manually.
# # If you use Kaggle API or have files already at DATA_PATH, skip this cell.

# from google.colab import files

# os.makedirs(cfg.DATA_PATH, exist_ok=True)
# print(f"Upload the following three files for dataset '{cfg.DATASET_NAME}':")
# print(f"  {cfg.DATASET_NAME}_Trans.csv")
# print(f"  {cfg.DATASET_NAME}_accounts.csv")
# print(f"  {cfg.DATASET_NAME}_Patterns.txt")

# uploaded = files.upload()
# for filename in uploaded.keys():
#     dst = os.path.join(cfg.DATA_PATH, filename)
#     with open(dst, "wb") as f:
#         f.write(uploaded[filename])
#     print(f"Saved: {dst}")

In [ ]:
# def load_dataset(cfg):
#     base = os.path.join(cfg.DATA_PATH, cfg.DATASET_NAME)
#     print(f"\nLoading {cfg.DATASET_NAME} ...")

#     trans_df    = pd.read_csv(f"{base}_Trans.csv")
#     accounts_df = pd.read_csv(f"{base}_accounts.csv")

#     patterns_path = f"{base}_Patterns.txt"
#     patterns_text = open(patterns_path).read() if os.path.exists(patterns_path) else ""

#     print(f"  Transactions : {trans_df.shape}")
#     print(f"  Accounts     : {accounts_df.shape}")
#     print(f"  Patterns file: {'found' if patterns_text else 'not found'}")
#     return trans_df, accounts_df, patterns_text

# trans_df, accounts_df, patterns_text = load_dataset(cfg)
# print("\nTransaction columns:")
# print(list(trans_df.columns))
# print("\nAccount columns:")
# print(list(accounts_df.columns))
# print("\nSample transactions:")
# trans_df.head(3)

In [7]:
def preprocess(trans_df, accounts_df):
    """
    Column schema
    -------------
    Trans   : Timestamp, From Bank, Account, To Bank, Account.1,
              Amount Received, Receiving Currency, Amount Paid,
              Payment Currency, Payment Format, Is Laundering
    Accounts: Bank ID, Account Number, Entity ID, Entity Name
    """
    print("Preprocessing ...")

    # ---- account id = "BankID_AccountNumber" ----
    acc_col   = accounts_df.columns[1]   # Account Number
    bank_col  = accounts_df.columns[0]   # Bank ID
    accounts_df["acc_id"] = (
        accounts_df[bank_col].astype(str) + "_" +
        accounts_df[acc_col].astype(str)
    )

    df = trans_df.copy()

    # Build consistent account IDs from transaction table
    df["From_Account"] = df["From Bank"].astype(str) + "_" + df["Account"].astype(str)
    df["To_Account"]   = df["To Bank"].astype(str)   + "_" + df["Account.1"].astype(str)

    # Timestamp
    df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")
    df = df.dropna(subset=["From_Account", "To_Account", "Timestamp"])

    # Label
    if "Is Laundering" in df.columns:
        df["label"] = df["Is Laundering"].astype(int)
    else:
        df["label"] = 0

    
    # Payment format encoding
    le = LabelEncoder()
    df["payment_enc"] = le.fit_transform(df["Payment Format"].astype(str))

    # Currency encoding
    le2 = LabelEncoder()
    df["currency_enc"] = le2.fit_transform(df["Payment Currency"].astype(str))

    # Numeric features for edges
    df["amount_log"]      = np.log1p(df["Amount Paid"].fillna(0))
    df["hour"]            = df["Timestamp"].dt.hour / 23.0
    df["day_of_week"]     = df["Timestamp"].dt.dayofweek / 6.0
    df["unix_ts"]         = df["Timestamp"].astype(np.int64) / 1e18   # normalised

    df = df.reset_index(drop=True)

    print(f"  Cleaned transactions : {len(df)}")
    print(f"  Laundering rate      : {df['label'].mean():.5f}")

    return df, le, le2

trans_clean, le_payment, le_currency = preprocess(trans_df, accounts_df)

Preprocessing ...
  Cleaned transactions : 451092
  Laundering rate      : 0.50000


In [8]:
def build_graph(df):
    """
    Node  = unique bank account
    Edge  = transaction (directed, From -> To)
    Edge features: [amount_log, hour, day_of_week, payment_enc_norm, currency_enc_norm]
    Node features: aggregated statistics over incident transactions
    Labels are on edges.
    
    Because full-graph training is used (no LinkNeighborLoader),
    this works without pyg-lib or torch-sparse.
    """
    print("Building graph ...")

    # ---- node index mapping ----
    all_accounts = pd.unique(
        pd.concat([df["From_Account"], df["To_Account"]])
    )
    node2idx = {acc: i for i, acc in enumerate(all_accounts)}
    idx2node = {i: acc for acc, i in node2idx.items()}
    num_nodes = len(node2idx)

    src = df["From_Account"].map(node2idx).values.astype(np.int64)
    dst = df["To_Account"].map(node2idx).values.astype(np.int64)

    edge_index = torch.tensor(np.stack([src, dst], axis=0), dtype=torch.long)

    # ---- edge features ----
    num_pay  = df["payment_enc"].max() + 1
    num_cur  = df["currency_enc"].max() + 1

    edge_feat = np.stack([
        df["amount_log"].values,
        df["hour"].values,
        df["day_of_week"].values,
        df["payment_enc"].values / max(num_pay, 1),
        df["currency_enc"].values / max(num_cur, 1),
    ], axis=1).astype(np.float32)

    edge_attr = torch.tensor(edge_feat, dtype=torch.float)
    edge_y    = torch.tensor(df["label"].values, dtype=torch.long)

    # ---- node features: per-node stats (out-degree, in-degree, mean sent, mean received) ----
    node_feat = np.zeros((num_nodes, 4), dtype=np.float32)
    amounts   = df["amount_log"].values

    np.add.at(node_feat[:, 0], src, 1)             # out-degree count
    np.add.at(node_feat[:, 1], dst, 1)             # in-degree count
    np.add.at(node_feat[:, 2], src, amounts)       # sum sent
    np.add.at(node_feat[:, 3], dst, amounts)       # sum received

    # Normalise
    max_vals = node_feat.max(axis=0, keepdims=True) + 1e-8
    node_feat = node_feat / max_vals

    x = torch.tensor(node_feat, dtype=torch.float)

    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, edge_y=edge_y)
    data.num_nodes = num_nodes
    data.node2idx  = node2idx
    data.idx2node  = idx2node

    print(f"  Nodes        : {num_nodes:,}")
    print(f"  Edges        : {edge_index.shape[1]:,}")
    print(f"  Laundering   : {edge_y.sum().item():,}  ({edge_y.float().mean().item():.5f})")
    print(f"  Node feat dim: {x.shape[1]}")
    print(f"  Edge feat dim: {edge_attr.shape[1]}")
    return data

graph_data = build_graph(trans_clean)

Building graph ...
  Nodes        : 486,685
  Edges        : 451,092
  Laundering   : 225,546  (0.50000)
  Node feat dim: 4
  Edge feat dim: 5


In [9]:
def temporal_split(df, graph_data, train_frac=0.6, val_frac=0.2):
    """
    60 / 20 / 20 temporal split matching the paper's methodology.
    Splits on edge indices ordered by Timestamp.
    """
    n = len(df)
    sorted_idx = df["Timestamp"].argsort().values   # ascending time order

    t1 = int(n * train_frac)
    t2 = int(n * (train_frac + val_frac))

    train_idx = sorted_idx[:t1]
    val_idx   = sorted_idx[t1:t2]
    test_idx  = sorted_idx[t2:]

    def make_mask(indices):
        mask = torch.zeros(n, dtype=torch.bool)
        mask[indices] = True
        return mask

    graph_data.train_mask = make_mask(train_idx)
    graph_data.val_mask   = make_mask(val_idx)
    graph_data.test_mask  = make_mask(test_idx)

    print(f"Split  ->  train: {train_idx.shape[0]:,}  |  val: {val_idx.shape[0]:,}  |  test: {test_idx.shape[0]:,}")
    return graph_data

graph_data = temporal_split(trans_clean, graph_data)

Split  ->  train: 270,655  |  val: 90,218  |  test: 90,219


In [11]:
class EdgeGAT(nn.Module):
    """
    Graph Attention Network for edge (transaction) classification.

    Architecture
    ------------
    1. Two GAT layers produce node embeddings using node features.
    2. Each edge embedding is formed by concatenating:
         [src_node_emb || dst_node_emb || edge_attr]
    3. An MLP classifies each edge as laundering / normal.

    This architecture matches the GIN+EU approach referenced in the paper
    (edge updates on top of node message passing) and avoids LinkNeighborLoader,
    which requires pyg-lib or torch-sparse.
    """

    def __init__(self, node_feat_dim, edge_feat_dim, hidden_dim, num_classes,
                 num_layers=2, num_heads=4, dropout=0.3):
        super().__init__()

        self.gat_layers = nn.ModuleList()
        self.norms      = nn.ModuleList()

        in_dim = node_feat_dim
        for i in range(num_layers):
            out_dim  = hidden_dim // num_heads
            is_last  = (i == num_layers - 1)
            heads    = 1 if is_last else num_heads
            concat   = not is_last
            self.gat_layers.append(
                GATConv(in_dim, out_dim, heads=heads, concat=concat,
                        dropout=dropout, add_self_loops=True)
            )
            self.norms.append(nn.BatchNorm1d(out_dim * heads if concat else out_dim))
            in_dim = out_dim * heads if concat else out_dim

        node_emb_dim = in_dim

        # Edge MLP: src_emb + dst_emb + edge_feat
        mlp_in = node_emb_dim * 2 + edge_feat_dim
        self.edge_mlp = nn.Sequential(
            nn.Linear(mlp_in, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, num_classes),
        )

        self.dropout = dropout

    def get_node_embeddings(self, x, edge_index):
        for gat, norm in zip(self.gat_layers, self.norms):
            x = gat(x, edge_index)
            x = norm(x)
            x = F.elu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        return x

    def forward(self, x, edge_index, edge_attr):
        node_emb = self.get_node_embeddings(x, edge_index)

        src_idx = edge_index[0]
        dst_idx = edge_index[1]

        edge_emb = torch.cat([node_emb[src_idx], node_emb[dst_idx], edge_attr], dim=-1)
        logits   = self.edge_mlp(edge_emb)
        return logits, node_emb


# Instantiate
model = EdgeGAT(
    node_feat_dim = graph_data.x.shape[1],
    edge_feat_dim = graph_data.edge_attr.shape[1],
    hidden_dim    = cfg.HIDDEN_DIM,
    num_classes   = 2,
    num_layers    = cfg.NUM_LAYERS,
    num_heads     = cfg.NUM_HEADS,
    dropout       = cfg.DROPOUT,
).to(cfg.DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nTrainable parameters: {total_params:,}")

EdgeGAT(
  (gat_layers): ModuleList(
    (0): GATConv(4, 16, heads=4)
    (1): GATConv(64, 16, heads=1)
  )
  (norms): ModuleList(
    (0): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (1): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (edge_mlp): Sequential(
    (0): Linear(in_features=37, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): ReLU()
    (6): Linear(in_features=32, out_features=2, bias=True)
  )
)

Trainable parameters: 6,386


In [12]:
def compute_class_weight(edge_y):
    n_neg = (edge_y == 0).sum().item()
    n_pos = (edge_y == 1).sum().item()
    total = n_neg + n_pos
    # inverse frequency
    w_neg = total / (2 * n_neg + 1e-8)
    w_pos = total / (2 * n_pos + 1e-8)
    # additional manual upweight for minority
    w_pos *= cfg.MINORITY_CLASS_WEIGHT
    return torch.tensor([w_neg, w_pos], dtype=torch.float)


class_weights = compute_class_weight(graph_data.edge_y).to(cfg.DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=cfg.LEARNING_RATE,
                             weight_decay=cfg.WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=5
)

print(f"Class weights: neg={class_weights[0]:.3f}  pos={class_weights[1]:.3f}")


def move_graph_to_device(data, device):
    data.x          = data.x.to(device)
    data.edge_index = data.edge_index.to(device)
    data.edge_attr  = data.edge_attr.to(device)
    data.edge_y     = data.edge_y.to(device)
    return data


def train_epoch(model, data, mask, optimizer, criterion, device):
    model.train()
    optimizer.zero_grad()
    logits, _ = model(data.x, data.edge_index, data.edge_attr)
    loss = criterion(logits[mask], data.edge_y[mask])
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    return loss.item()


@torch.no_grad()
def evaluate(model, data, mask, device):
    model.eval()
    logits, _ = model(data.x, data.edge_index, data.edge_attr)
    loss   = criterion(logits[mask], data.edge_y[mask]).item()
    probs  = F.softmax(logits[mask], dim=-1)[:, 1].cpu().numpy()
    labels = data.edge_y[mask].cpu().numpy()
    preds  = (probs >= cfg.SUSPICION_THRESHOLD).astype(int)
    f1     = f1_score(labels, preds, zero_division=0)
    return loss, f1, preds, probs

Class weights: neg=1.000  pos=10.000


In [13]:
# Move full graph to GPU once; full-graph forward pass — no DataLoader needed,
# so no pyg-lib / torch-sparse dependency.
graph_data = move_graph_to_device(graph_data, cfg.DEVICE)

history = {"train_loss": [], "val_loss": [], "val_f1": []}
best_val_f1   = 0.0
patience_ctr  = 0
best_state    = None
save_path = os.path.join("/kaggle/working/", f"model_{cfg.DATASET_NAME}.pth")
print(f"Starting full-graph training on {cfg.DEVICE}  ({cfg.EPOCHS} epochs)\n")

for epoch in trange(cfg.EPOCHS, desc="Training"):
    train_loss = train_epoch(
        model, graph_data, graph_data.train_mask, optimizer, criterion, cfg.DEVICE
    )
    val_loss, val_f1, _, _ = evaluate(
        model, graph_data, graph_data.val_mask, cfg.DEVICE
    )

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_f1"].append(val_f1)

    scheduler.step(val_f1)

    if val_f1 > best_val_f1:
        best_val_f1  = val_f1
        best_state   = {k: v.clone() for k, v in model.state_dict().items()}
        patience_ctr = 0
    else:
        patience_ctr += 1

    if (epoch + 1) % 5 == 0:
        print(f"  Epoch {epoch+1:3d} | train_loss={train_loss:.4f} | "
              f"val_loss={val_loss:.4f} | val_f1={val_f1:.4f} | "
              f"best_val_f1={best_val_f1:.4f}")

    if patience_ctr >= cfg.EARLY_STOPPING_PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

# Restore best checkpoint
if best_state is not None:
    model.load_state_dict(best_state,save_path)

print(f"\nBest Validation F1: {best_val_f1:.4f}")

Starting full-graph training on cuda  (50 epochs)



Training:   0%|          | 0/50 [00:00<?, ?it/s]

  Epoch   5 | train_loss=0.5234 | val_loss=0.4708 | val_f1=0.6900 | best_val_f1=0.6900
  Epoch  10 | train_loss=0.4743 | val_loss=0.4277 | val_f1=0.6900 | best_val_f1=0.6900

Early stopping at epoch 11

Best Validation F1: 0.6900


In [14]:
test_loss, test_f1, test_preds, test_probs = evaluate(
    model, graph_data, graph_data.test_mask, cfg.DEVICE
)
test_labels = graph_data.edge_y[graph_data.test_mask].cpu().numpy()

precision = precision_score(test_labels, test_preds, zero_division=0)
recall    = recall_score(test_labels, test_preds, zero_division=0)
try:
    auc = roc_auc_score(test_labels, test_probs)
except ValueError:
    auc = float("nan")

print("=" * 50)
print("TEST SET RESULTS")
print("=" * 50)
print(f"  F1 Score  : {test_f1:.4f}")
print(f"  Precision : {precision:.4f}")
print(f"  Recall    : {recall:.4f}")
print(f"  ROC-AUC   : {auc:.4f}")
print(f"  Loss      : {test_loss:.4f}")
print("=" * 50)

TEST SET RESULTS
  F1 Score  : 0.7770
  Precision : 0.6353
  Recall    : 1.0000
  ROC-AUC   : 0.6479
  Loss      : 0.5494


In [28]:
def identify_ibm_pattern(acc_id, transactions):
    """
    Analyzes transaction topology to map to IBM's 8 laundering patterns.
    """
    # Identify unique counterparties
    senders = set(tx['sender_id'] for tx in transactions if tx['sender_id'] != acc_id)
    receivers = set(tx['receiver_id'] for tx in transactions if tx['receiver_id'] != acc_id)
    
    in_degree = len(senders)
    out_degree = len(receivers)
    
    # 1. Simple Cycle: A -> B -> A
    if any(tx['sender_id'] == tx['receiver_id'] for tx in transactions):
        return "Simple Cycle"
    
    # 2. Fan-out: One source to many destinations
    if in_degree == 1 and out_degree > 3:
        return "Fan-out"
    
    # 3. Fan-in: Many sources to one destination
    if in_degree > 3 and out_degree == 1:
        return "Fan-in"
    
    # 4. Gather-Scatter: Many in, then many out
    if in_degree > 2 and out_degree > 2:
        return "Gather-scatter"
    
    # 5. Scatter-Gather: Out to many, then back to one
    if in_degree > 3 and out_degree > 3:
        return "Scatter-gather"
        
    # 6. Bipartite: Two distinct groups (Money Mules)
    if in_degree >= 5 and out_degree >= 5:
        return "Bipartite"
        
    # 7. Stack: Layering through multiple accounts
    if len(transactions) > 10 and in_degree > 1:
        return "Stack"
        
    # 8. Random: High volume but no clear structure
    return "Random"

In [29]:
@torch.no_grad()
def get_all_probs(model, data, device):
    model.eval()
    logits, node_emb = model(data.x, data.edge_index, data.edge_attr)
    probs = F.softmax(logits, dim=-1)[:, 1].cpu().numpy()
    return probs, node_emb.cpu().numpy()

all_probs, node_emb = get_all_probs(model, graph_data, cfg.DEVICE)

# Flag suspicious edges
suspicious_mask_np = all_probs >= cfg.SUSPICION_THRESHOLD
suspicious_indices = np.where(suspicious_mask_np)[0]
print(f"Suspicious transactions flagged: {suspicious_mask_np.sum():,}")

# Build account-level scores and transaction lists
account_scores       = defaultdict(float)
account_trans_count  = defaultdict(int)
account_transactions = defaultdict(list)

idx2node = graph_data.idx2node if hasattr(graph_data, "idx2node") else {}

src_all = graph_data.edge_index[0].cpu().numpy()
dst_all = graph_data.edge_index[1].cpu().numpy()

for edge_idx in tqdm(suspicious_indices, desc="Building account scores"):
    src_node = idx2node.get(int(src_all[edge_idx]), f"ACC_{src_all[edge_idx]}")
    dst_node = idx2node.get(int(dst_all[edge_idx]), f"ACC_{dst_all[edge_idx]}")
    prob     = float(all_probs[edge_idx])

    account_scores[src_node] += prob
    account_scores[dst_node] += prob
    account_trans_count[src_node] += 1
    account_trans_count[dst_node] += 1

    row = trans_clean.iloc[edge_idx]
    record = {
        "transaction_id": f"T{edge_idx}",
        "sender_id"     : src_node,
        "receiver_id"   : dst_node,
        "amount"        : float(row["Amount Paid"]),
        "timestamp"     : str(row["Timestamp"]),
    }
    if len(account_transactions[src_node]) < 20:
        account_transactions[src_node].append(record)
    if len(account_transactions[dst_node]) < 20:
        account_transactions[dst_node].append(record)

# Normalise to [0, 100]
if account_scores:
    max_score = max(account_scores.values())
    account_scores = {
        k: min(100.0, (v / (max_score + 1e-8)) * 100)
        for k, v in account_scores.items()
    }


# --- STEP 1: DEFINE THE UPDATED FUNCTION ---
def detect_fraud_rings(suspicious_edge_idx, src_all, dst_all, idx2node, account_scores, account_transactions):
    # Directed adjacency for flow analysis (who sent to whom)
    adj_directed = defaultdict(list)
    # Undirected adjacency for finding the cluster/ring
    adj_undirected = defaultdict(set)
    
    for eidx in suspicious_edge_idx:
        s = idx2node.get(int(src_all[eidx]), f"ACC_{src_all[eidx]}")
        d = idx2node.get(int(dst_all[eidx]), f"ACC_{dst_all[eidx]}")
        adj_directed[s].append(d)
        adj_undirected[s].add(d)
        adj_undirected[d].add(s)

    visited = set()
    rings = {}
    ring_id = 0
    
    for start_node in adj_undirected:
        if start_node in visited: continue
        members = []
        queue = deque([start_node])
        visited.add(start_node)
        while queue:
            node = queue.popleft()
            members.append(node)
            for nb in adj_undirected[node]:
                if nb not in visited:
                    visited.add(nb)
                    queue.append(nb)
        
        if len(members) >= 2:
            # --- START/END NODE LOGIC ---
            in_counts = {m: 0 for m in members}
            out_counts = {m: 0 for m in members}
            
            for m in members:
                for neighbor in adj_directed[m]:
                    if neighbor in in_counts:
                        out_counts[m] += 1
                        in_counts[neighbor] += 1
            
            # Origin = highest outward flow | Exit = highest inward flow
            origin_node = max(out_counts, key=out_counts.get)
            exit_node = max(in_counts, key=in_counts.get)

            ring_txs = []
            for m in members:
                ring_txs.extend(account_transactions.get(m, []))
            
            pattern = identify_ibm_pattern(members[0], ring_txs)
            risk = float(np.mean([account_scores.get(m, 0) for m in members]))
            
            rings[f"RING_{ring_id:03d}"] = {
                "member_accounts": members,
                "pattern_type": pattern,
                "risk_score": risk,
                "origin": origin_node,
                "destination": exit_node
            }
            ring_id += 1
            
    return rings

# Call the function
fraud_rings = detect_fraud_rings(
    suspicious_indices, src_all, dst_all, idx2node, account_scores, account_transactions
)
# --- STEP 2: CALL THE FUNCTION CORRECTLY ---
# Note: I added 'account_transactions' as the last argument here
fraud_rings = detect_fraud_rings(
    suspicious_indices, 
    src_all, 
    dst_all, 
    idx2node, 
    account_scores, 
    account_transactions
)

print(f"Fraud rings detected: {len(fraud_rings):,}")


# fraud_rings = detect_fraud_rings(
#     suspicious_indices, src_all, dst_all, idx2node, account_scores
# )
# print(f"Fraud rings detected: {len(fraud_rings):,}")

Suspicious transactions flagged: 451,092


Building account scores:   0%|          | 0/451092 [00:00<?, ?it/s]

Fraud rings detected: 122,153


In [33]:
def generate_report(account_scores, account_transactions, fraud_rings, graph_data, cfg):
    t0 = time.time()
    
    # Process Rings into the new format
    formatted_rings = []
    for rid, info in fraud_rings.items():
        # Get member details with dynamic roles
        ring_accounts = []
        for acc_id in info["member_accounts"]:
            # Determine Role
            if acc_id == info["origin"]:
                role = "Originator"
            elif acc_id == info["destination"]:
                role = "Exit Point"
            else:
                role = "Intermediary / Mule"
                
            # Get transactions and patterns for this account
            acc_txs = account_transactions.get(acc_id, [])
            pattern_list = [info["pattern_type"]]
            if len(acc_txs) > 10: pattern_list.append("high_velocity")
            
            score = account_scores.get(acc_id, 0.0)
            if score > 80: pattern_list.append("high_risk_score")

            ring_accounts.append({
                "account_id": acc_id,
                "role": role,
                "suspicion_score": round(score, 2),
                "detected_patterns": pattern_list
            })

        # Get transaction sample for this ring
        ring_tx_sample = []
        for m in info["member_accounts"]:
            ring_tx_sample.extend(account_transactions.get(m, []))

        formatted_rings.append({
            "ring_id": rid,
            "flow_analysis": {
                "origin_node": info["origin"],
                "exit_node": info["destination"],
                "total_hop_count": len(info["member_accounts"]) - 1
            },
            "accounts": ring_accounts,
            "transactions": ring_tx_sample[:10], # Top 10 transactions for context
            "context": {
                "time_window": datetime.now().strftime("%Y-%m-%d"),
                "analysis_type": "fraud_ring_explanation",
                "pattern_identified": info["pattern_type"]
            }
        })

    # Prepare final structure
    # Note: suspicious_accounts is kept for compatibility with your preview code
    suspicious_accounts = [acc for ring in formatted_rings for acc in ring["accounts"] if acc["suspicion_score"] > cfg.SUSPICION_THRESHOLD*100]
    suspicious_accounts.sort(key=lambda x: x["suspicion_score"], reverse=True)

    report = {
        "suspicious_accounts": suspicious_accounts,
        "fraud_rings": formatted_rings,
        "summary": {
            "total_accounts_analyzed": graph_data.num_nodes,
            "fraud_rings_detected": len(fraud_rings),
            "processing_time_seconds": round(time.time() - t0, 3),
            "dataset": cfg.DATASET_NAME
        }
    }
    return report

# Generate the aml_report object
aml_report = generate_report(account_scores, account_transactions, fraud_rings, graph_data, cfg)
# Execute the updated report
# aml_report = generate_report(
#     account_scores, account_transactions, fraud_rings,
#     graph_data, cfg, trans_clean
# )

print("=" * 60)
print("AML DETECTION REPORT - SUMMARY")
print("=" * 60)
print(json.dumps(aml_report["summary"], indent=2))

print("\nTop 5 Suspicious Accounts (from Rings):")
# In the new schema, we pull accounts from the fraud_rings directly
for ring in aml_report["fraud_rings"][:5]:
    for acc in ring["accounts"][:1]: # Show the primary account for each top ring
        print(f"  {acc['account_id']} | Score: {acc['suspicion_score']} | "
              f"Role: {acc['role']} | Patterns: {acc['detected_patterns']}")

print("\nTop 5 Fraud Rings (Flow Analysis):")
for ring in aml_report["fraud_rings"][:5]:
    flow = ring["flow_analysis"]
    print(f"  {ring['ring_id']} | {flow['origin_node']} -> {flow['exit_node']} | "
          f"Pattern: {ring['context']['pattern_identified']}")

AML DETECTION REPORT - SUMMARY
{
  "total_accounts_analyzed": 486685,
  "fraud_rings_detected": 122153,
  "processing_time_seconds": 4.703,
  "dataset": "HI-Large"
}

Top 5 Suspicious Accounts (from Rings):
  17644_814A370B0 | Score: 0.01 | Role: Intermediary / Mule | Patterns: ['Simple Cycle']
  4214_801C65640 | Score: 0.02 | Role: Intermediary / Mule | Patterns: ['Simple Cycle']
  4_81A6BF760 | Score: 0.01 | Role: Originator | Patterns: ['Random']
  110478_8117A6D70 | Score: 0.02 | Role: Originator | Patterns: ['Random']
  229256_831E42B20 | Score: 0.01 | Role: Originator | Patterns: ['Random']

Top 5 Fraud Rings (Flow Analysis):
  RING_000 | 70_100428660 -> 272896_81BBEA160 | Pattern: Simple Cycle
  RING_001 | 25642_802702F10 -> 17569_809EA3DB0 | Pattern: Simple Cycle
  RING_002 | 4_81A6BF760 -> 277486_81C633BE0 | Pattern: Random
  RING_003 | 110478_8117A6D70 -> 240503_833E76AA0 | Pattern: Random
  RING_004 | 229256_831E42B20 -> 5560_83993B290 | Pattern: Random


In [31]:
print("=" * 60)
print("FRAUD FLOW ANALYSIS")
print("=" * 60)

# Sort by risk and show the flow
for rid, info in list(fraud_rings.items())[:10]:
    print(f"Ring: {rid}")
    print(f"  Pattern : {info['pattern_type']}")
    print(f"  Origin  : {info['origin']} (Funds start here)")
    print(f"  Exit    : {info['destination']} (Funds leave here)")
    print(f"  Members : {len(info['member_accounts'])}")
    print("-" * 30)

FRAUD FLOW ANALYSIS
Ring: RING_000
  Pattern : Simple Cycle
  Origin  : 70_100428660 (Funds start here)
  Exit    : 272896_81BBEA160 (Funds leave here)
  Members : 174509
------------------------------
Ring: RING_001
  Pattern : Simple Cycle
  Origin  : 25642_802702F10 (Funds start here)
  Exit    : 17569_809EA3DB0 (Funds leave here)
  Members : 17
------------------------------
Ring: RING_002
  Pattern : Random
  Origin  : 4_81A6BF760 (Funds start here)
  Exit    : 277486_81C633BE0 (Funds leave here)
  Members : 2
------------------------------
Ring: RING_003
  Pattern : Random
  Origin  : 110478_8117A6D70 (Funds start here)
  Exit    : 240503_833E76AA0 (Funds leave here)
  Members : 3
------------------------------
Ring: RING_004
  Pattern : Random
  Origin  : 229256_831E42B20 (Funds start here)
  Exit    : 5560_83993B290 (Funds leave here)
  Members : 2
------------------------------
Ring: RING_005
  Pattern : Simple Cycle
  Origin  : 11796_80B931800 (Funds start here)
  Exit    : 1

In [ ]:
out_path = f"/kaggle/working/aml_report_{cfg.DATASET_NAME}.json"
with open(out_path, "w") as f:
    json.dump(aml_report, f, indent=2, default=str)

print(f"Report saved to: {out_path}")
print("\nFULL JSON REPORT (first 3 suspicious accounts and first ring):")

preview = dict(aml_report)
# Slicing the lists for a clean preview
preview["suspicious_accounts"] = aml_report["suspicious_accounts"][:3]
preview["fraud_rings"]         = aml_report["fraud_rings"][:1] # Showing 1 ring to see the new structure

print(json.dumps(preview, indent=2, default=str))

Report saved to: /kaggle/working/aml_report_HI-Large.json

FULL JSON REPORT (first 3 suspicious accounts and first ring):


In [27]:
# t Save the Trained Model
import os

# Define the save path
save_path = os.path.join("/kaggle/working/", f"model_{cfg.DATASET_NAME}2.pth")

# Save the best_state captured during training
if best_state is not None:
    torch.save(best_state, save_path)
    print(f"Model saved successfully to: {save_path}")
else:
    print("Error: No trained model state found to save.")

Model saved successfully to: /kaggle/working/model_HI-Large2.pth
